In [ ]:
"""
Machine Learning Analysis of Kohomono Phonology Data
PhD thesis: "A Phonology of Kohomono" by Etu, Mercy Runyi
University of Port Harcourt
Manuscript for Code
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
DPI = 600
FIGDIR = "kohomono_figures"
import os; os.makedirs(FIGDIR, exist_ok=True)

# ─── KOHOMONO PHONOLOGICAL DATA ────────────────────────────────────────────────
# Based on Etu (2020) primary field data from Kohomono language, Cross River State

# Consonant inventory with phonological features
consonants_data = {
    'phoneme': ['p','b','t','d','k','g','kw','gw','kp','gb','dʒ',
                'm','n','ɲ','ŋ','f','v','s','z','ʃ','h','ɹ','j','w'],
    'voiced':  [0,  1,  0,  1,  0,  1,  0,   1,  0,  1,  1,
                1,  1,  1,  1,  0,  1,  0,  1,  0,  0,  1,  1,  1],
    'manner':  ['stop','stop','stop','stop','stop','stop','stop','stop','stop','stop','affricate',
                'nasal','nasal','nasal','nasal','fricative','fricative','fricative','fricative','fricative','fricative','approximant','approximant','approximant'],
    'place':   ['bilabial','bilabial','alveolar','alveolar','velar','velar','labiovelar','labiovelar','labio-velar','labio-velar','palatal',
                'bilabial','alveolar','palatal','velar','labiodental','labiodental','alveolar','alveolar','palato-alveolar','glottal','alveolar','palatal','bilabial'],
    'labial':  [1,1,0,0,0,0,1,1,1,1,0,  1,0,0,0,1,1,0,0,0,0,0,0,1],
    'coronal': [0,0,1,1,0,0,0,0,0,0,1,  0,1,1,0,0,0,1,1,1,0,1,1,0],
    'dorsal':  [0,0,0,0,1,1,1,1,1,1,0,  0,0,0,1,0,0,0,0,0,0,0,0,0],
    'sonorant':[0,0,0,0,0,0,0,0,0,0,0,  1,1,1,1,0,0,0,0,0,0,1,1,1],
    'continuant':[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1],
    'nasal_feat':[0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0],
    'word_initial':[1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,0,0,1,1],
    'word_medial': [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    'word_final':  [1,1,1,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0],
    'has_allophones':[1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0],
    'freq_score':  [3,5,4,3,4,2,3,3,4,3,2,  5,5,4,3,4,3,4,3,2,3,4,3,3],
}

df_cons = pd.DataFrame(consonants_data)

# Vowel inventory with features
vowels_data = {
    'phoneme': ['i','e','ɛ','a','a:','u','o','o:','ɔ','ɔ:'],
    'height':  ['high','mid','mid-low','low','low','high','mid','mid','mid-low','mid-low'],
    'backness':['front','front','front','central','central','back','back','back','back','back'],
    'length':  ['short','short','short','short','long','short','short','long','short','long'],
    'ATR':     [1,1,0,0,0,1,1,1,0,0],
    'high_feat':[1,0,0,0,0,1,0,0,0,0],
    'low_feat': [0,0,0,1,1,0,0,0,0,0],
    'back_feat':[0,0,0,0,0,1,1,1,1,1],
    'long_feat':[0,0,0,0,1,0,0,1,0,1],
    'freq_score':[5,4,4,5,2,4,4,2,4,2],
}
df_vowels = pd.DataFrame(vowels_data)

# Tone data - lexical minimal pairs showing tonal contrast
tone_data = {
    'word_pair': ['kpè/kpé','gwá/gwá','ìʃín/ìʃín','kpé/kpé','bàn/bàn','jén/jém',
                  'ézèm/ézèn','tàn/dàn','bà/pà','kù/ɡú','àkà/áɡà','ókó/óɡó',
                  'úpà/úkpà','ótà/ódà','ótàm/ódàm'],
    'tone1': ['L','H','LH','H','L','H','LH','L','L','L','LL','HH','HH','HH','HHH'],
    'tone2': ['H','H','LH','H','L','H','LH','L','L','H','HH','HH','HH','HH','HHH'],
    'meaning1': ['kneel','drink','hair','teach','marry','eye','crocodile','sell','women','in','mother','vegetable','calabash','string','traveller'],
    'meaning2': ['learn','cover','grass','learn/pay','smooth','take','meat','smooth','arrive','smoked','thread','in-law','valley','admit','be good'],
    'contrast_type': ['lexical','lexical','homophony','homophony','lexical','lexical','lexical','consonant','consonant','consonant','consonant','consonant','consonant','consonant','consonant'],
    'tone_level_diff': [1,0,0,0,0,0,0,0,0,1,1,0,1,1,1],
}
df_tone = pd.DataFrame(tone_data)

# Phonological processes data
processes_data = {
    'process': ['Consonant Assimilation','Gemination','Consonant Devoicing','Consonant Weakening',
                'Labialization','Palatalization','Nasalization','Vowel Harmony',
                'Vowel Deletion','Vowel Lengthening','Tonal Spreading','Tonal Delinking','Downstep'],
    'category': ['Consonant','Consonant','Consonant','Consonant',
                 'Consonant','Consonant','Vowel','Vowel',
                 'Vowel','Vowel','Tonal','Tonal','Tonal'],
    'direction': ['Progressive','Both','Regressive','Progressive',
                  'Progressive','Progressive','Both','Both',
                  'Regressive','Both','Both','Regressive','Progressive'],
    'frequency': [5,4,4,5,3,3,3,4,3,3,4,3,3],
    'environment': ['Word boundary','Word boundary','Final position','Intervocalic',
                    'Before back vowel','Before front vowel','Near nasals','Morpheme internal',
                    'Between vowels','Compensatory','Tone sequence','Vowel deletion','H-tone sequence'],
    'attested_count': [18,12,15,10,8,6,9,14,7,6,11,8,5],
}
df_proc = pd.DataFrame(processes_data)

# Syllable structure data
syllable_data = {
    'structure': ['V','CV','CVC','VCV','CVCV','CVVCV','N̩','N̩CV'],
    'count':     [12, 45, 38,  52,   87,   23,   8,    15],
    'examples':  ['ò (she)','sé (sit)','ʃìn (fifteen)','ónò (tears)','bùʃí (yam)','bóvén (thing)','N̩- (1sg)','N̩bà (I am back)'],
    'word_class':['pronoun','verb','noun','noun','noun','noun','prefix','sentence'],
    'open_closed':['open','open','closed','open','open','open','syllabic','closed'],
}
df_syll = pd.DataFrame(syllable_data)

print("Data loaded. Consonants:", len(df_cons), "| Vowels:", len(df_vowels))
print("Processes:", len(df_proc), "| Tone pairs:", len(df_tone))

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Kohomono Consonant Inventory Overview
# ═══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('white')
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel A: Consonants by Manner
ax1 = fig.add_subplot(gs[0, 0])
manner_counts = df_cons['manner'].value_counts()
colors_manner = ['#1D3557','#457B9D','#A8DADC','#E63946','#F4A261','#2A9D8F']
bars = ax1.barh(manner_counts.index, manner_counts.values, color=colors_manner[:len(manner_counts)], 
                edgecolor='white', linewidth=1.2, height=0.65)
for bar, val in zip(bars, manner_counts.values):
    ax1.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'n={val}', 
             va='center', fontsize=10, fontweight='bold')
ax1.set_title('A. Consonants by Manner of Articulation', fontsize=11, fontweight='bold', pad=8)
ax1.set_xlabel('Count', fontsize=10)
ax1.set_xlim(0, 13)
ax1.spines[['top','right']].set_visible(False)
ax1.set_facecolor('#F8F9FA')

# Panel B: Consonants by Place
ax2 = fig.add_subplot(gs[0, 1])
place_counts = df_cons['place'].value_counts()
colors_place = ['#E63946','#457B9D','#F4A261','#2A9D8F','#264653','#A8DADC','#E9C46A']
bars2 = ax2.barh(place_counts.index, place_counts.values, color=colors_place[:len(place_counts)], 
                 edgecolor='white', linewidth=1.2, height=0.65)
for bar, val in zip(bars2, place_counts.values):
    ax2.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'n={val}', 
             va='center', fontsize=10, fontweight='bold')
ax2.set_title('B. Consonants by Place of Articulation', fontsize=11, fontweight='bold', pad=8)
ax2.set_xlabel('Count', fontsize=10)
ax2.set_xlim(0, 10)
ax2.spines[['top','right']].set_visible(False)
ax2.set_facecolor('#F8F9FA')

# Panel C: Voiced vs Voiceless
ax3 = fig.add_subplot(gs[0, 2])
voice_counts = df_cons['voiced'].value_counts()
labels = ['Voiced', 'Voiceless']
sizes = [voice_counts.get(1, 0), voice_counts.get(0, 0)]
wedge_colors = ['#E63946', '#457B9D']
wedges, texts, autotexts = ax3.pie(sizes, labels=labels, colors=wedge_colors, autopct='%1.1f%%',
                                    startangle=140, textprops={'fontsize': 11},
                                    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold')
ax3.set_title('C. Voicing Distribution\n(24 consonant phonemes)', fontsize=11, fontweight='bold', pad=8)

# Panel D: Positional Distribution of Consonants
ax4 = fig.add_subplot(gs[1, 0])
positions = ['Word Initial', 'Word Medial', 'Word Final']
pos_counts = [df_cons['word_initial'].sum(), df_cons['word_medial'].sum(), df_cons['word_final'].sum()]
bar_pos = ax4.bar(positions, pos_counts, color=['#1D3557','#457B9D','#A8DADC'], 
                   edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bar_pos, pos_counts):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'n={val}', 
             ha='center', fontsize=11, fontweight='bold')
ax4.set_title('D. Consonant Positional Distribution', fontsize=11, fontweight='bold', pad=8)
ax4.set_ylabel('Number of Phonemes', fontsize=10)
ax4.set_ylim(0, 28)
ax4.spines[['top','right']].set_visible(False)
ax4.set_facecolor('#F8F9FA')

# Panel E: Vowel inventory positions
ax5 = fig.add_subplot(gs[1, 1])
vowel_plot = {
    'i': (0.1, 0.95), 'e': (0.15, 0.6), 'ɛ': (0.2, 0.35),
    'a': (0.5, 0.05), 'a:': (0.52, 0.05),
    'u': (0.9, 0.95), 'o': (0.85, 0.6), 'o:': (0.87, 0.6),
    'ɔ': (0.8, 0.35), 'ɔ:': (0.82, 0.35)
}
for v, (x, y) in vowel_plot.items():
    color = '#E63946' if ':' in v else '#1D3557'
    size = 180 if ':' in v else 220
    ax5.scatter(x, y, s=size, color=color, zorder=5, edgecolors='white', linewidth=1.5)
    ax5.text(x, y+0.055, f'/{v}/', ha='center', va='bottom', fontsize=11, 
             fontweight='bold', color=color)
# Draw vowel trapezoid outline
ax5.plot([0.05, 0.95], [1.0, 1.0], 'gray', linewidth=1, alpha=0.4, linestyle='--')
ax5.plot([0.05, 0.5],  [1.0, 0.0], 'gray', linewidth=1, alpha=0.4, linestyle='--')
ax5.plot([0.95, 0.5],  [1.0, 0.0], 'gray', linewidth=1, alpha=0.4, linestyle='--')
ax5.set_xlim(-0.05, 1.1); ax5.set_ylim(-0.1, 1.15)
ax5.set_xticks([0.05, 0.5, 0.95]); ax5.set_xticklabels(['Front','Central','Back'], fontsize=10)
ax5.set_yticks([0.0, 0.35, 0.6, 0.95]); ax5.set_yticklabels(['Low','Mid-Low','Mid','High'], fontsize=10)
ax5.set_title('E. Vowel Phoneme Space\n(7 short + 3 long = 10 vowel phonemes)', fontsize=11, fontweight='bold', pad=8)
legend_els = [mpatches.Patch(color='#1D3557', label='Short vowels (7)'),
              mpatches.Patch(color='#E63946', label='Long vowels (3)')]
ax5.legend(handles=legend_els, fontsize=9, loc='lower right')
ax5.set_facecolor('#F8F9FA')

# Panel F: Phonological feature heatmap
ax6 = fig.add_subplot(gs[1, 2])
feat_cols = ['voiced','labial','coronal','dorsal','sonorant','continuant','nasal_feat']
feat_labels = ['Voiced','Labial','Coronal','Dorsal','Sonorant','Continuant','Nasal']
feat_matrix = df_cons[feat_cols].values.T
sns.heatmap(feat_matrix, ax=ax6, cmap='Blues', xticklabels=df_cons['phoneme'],
            yticklabels=feat_labels, cbar=False, linewidths=0.3, linecolor='white',
            annot=False)
ax6.set_title('F. Distinctive Feature Matrix\n(24 consonant phonemes)', fontsize=11, fontweight='bold', pad=8)
ax6.set_xticklabels(df_cons['phoneme'], fontsize=8, rotation=0)
ax6.set_yticklabels(feat_labels, fontsize=9, rotation=0)


plt.savefig(f'{FIGDIR}/fig1_phoneme_inventory.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 1")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Machine Learning Phoneme Classification
# ═══════════════════════════════════════════════════════════════════════════════
# Classify consonants into manner classes using distinctive features
feature_cols = ['voiced','labial','coronal','dorsal','sonorant','continuant','nasal_feat','word_initial','word_medial','word_final']
X = df_cons[feature_cols].values
y_manner = df_cons['manner'].values
le = LabelEncoder()
y_enc = le.fit_transform(y_manner)

# Use all data with cross-validation (small dataset)
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Reg.': LogisticRegression(max_iter=1000, random_state=42, C=1.0),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42, probability=True),
    'k-NN': KNeighborsClassifier(n_neighbors=3),
    'Grad. Boost': GradientBoostingClassifier(n_estimators=80, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

model_names_short = ['RF','LR','SVM','k-NN','GB']
bar_colors = ['#1D3557','#457B9D','#E63946','#F4A261','#2A9D8F']
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y_enc, cv=skf, scoring='accuracy')
    cv_results[name] = scores

# Panel A: CV Accuracy
means = [cv_results[n].mean() for n in models]
stds  = [cv_results[n].std()  for n in models]
bars = axes[0].bar(model_names_short, means, yerr=stds, color=bar_colors, 
                    capsize=6, edgecolor='white', linewidth=1.2, width=0.6,
                    error_kw={'elinewidth':2,'ecolor':'#333'})
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('A. Consonant Class Classification\n(4-Fold CV Accuracy ± SD)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_ylim(0, 1.15)
axes[0].axhline(0.8, color='crimson', linestyle='--', alpha=0.5, linewidth=1.5, label='0.80 baseline')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Feature importance from RF
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X, y_enc)
feat_imp = rf_model.feature_importances_
feat_labels_short = ['Voiced','Labial','Coronal','Dorsal','Sonorant','Continuant','Nasal','Init.','Med.','Final']
sorted_idx = np.argsort(feat_imp)[::-1]
axes[1].barh([feat_labels_short[i] for i in sorted_idx], feat_imp[sorted_idx],
              color=['#E63946' if i < 3 else '#457B9D' for i in range(len(sorted_idx))],
              edgecolor='white', linewidth=1, height=0.6)
axes[1].set_title('B. Feature Importance\n(Random Forest, Consonant Classification)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xlabel('Importance Score', fontsize=11)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')

# Panel C: PCA of consonant feature space
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

manner_colors = {'stop':'#1D3557','nasal':'#E63946','fricative':'#F4A261',
                 'approximant':'#2A9D8F','affricate':'#A8DADC'}
for manner in df_cons['manner'].unique():
    mask = df_cons['manner'] == manner
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=manner_colors.get(manner,'gray'), s=120, label=manner.capitalize(),
                   edgecolors='white', linewidth=1.5, zorder=5, alpha=0.9)
for i, row in df_cons.iterrows():
    axes[2].annotate(f'/{row["phoneme"]}/', (X_pca[i,0], X_pca[i,1]),
                    textcoords='offset points', xytext=(4,4), fontsize=7.5, alpha=0.85)
axes[2].set_title(f'C. PCA of Consonant Feature Space\n(PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%})',
                  fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('PC1', fontsize=11); axes[2].set_ylabel('PC2', fontsize=11)
axes[2].legend(fontsize=8, loc='lower right', framealpha=0.9)
axes[2].spines[['top','right']].set_visible(False)
axes[2].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig2_ml_classification.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 2")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Cross-Validation and Model Performance
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel A: Violin plot of CV scores
fold_data = []
for name, short in zip(models.keys(), model_names_short):
    for score in cv_results[name]:
        fold_data.append({'Model': short, 'Accuracy': score})
fold_df = pd.DataFrame(fold_data)
sns.violinplot(data=fold_df, x='Model', y='Accuracy', palette=bar_colors, ax=axes[0],
               inner='point', linewidth=1.5)
axes[0].set_title('A. Cross-Validation Score Distributions\n(4-fold stratified CV)', fontsize=12, fontweight='bold', pad=8)
axes[0].set_ylabel('Classification Accuracy', fontsize=11)
axes[0].set_xlabel('Classifier', fontsize=11)
axes[0].axhline(0.8, color='crimson', linestyle='--', alpha=0.5, linewidth=1.5)
axes[0].set_ylim(0.3, 1.15)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Learning curves for best model (RF)
from sklearn.model_selection import learning_curve as lc_fn
# Simulate learning curve with extended synthetic augmented data
np.random.seed(99)
X_aug = np.vstack([X + np.random.normal(0, 0.08, X.shape) for _ in range(12)])
y_aug = np.tile(y_enc, 12)
train_sizes, train_scores, val_scores = lc_fn(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_aug, y_aug, cv=4, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1)
axes[1].plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#1D3557', lw=2.5, label='Training')
axes[1].fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1),
                      train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.15, color='#1D3557')
axes[1].plot(train_sizes, val_scores.mean(axis=1), 's-', color='#E63946', lw=2.5, label='Validation')
axes[1].fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1),
                      val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.15, color='#E63946')
axes[1].set_title('B. Learning Curve (Random Forest)\nConsonant Manner Classification', fontsize=12, fontweight='bold', pad=8)
axes[1].set_xlabel('Training Samples', fontsize=11); axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].legend(fontsize=10); axes[1].set_ylim(0.4, 1.05)
axes[1].grid(True, alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig3_cv_learning.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 3")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: Phonological Processes Analysis
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

# Panel A: Processes by category
cat_colors = {'Consonant':'#1D3557','Vowel':'#457B9D','Tonal':'#E63946'}
proc_cat = df_proc.groupby('category')['attested_count'].sum().reset_index()
bars = axes[0].bar(proc_cat['category'], proc_cat['attested_count'],
                    color=[cat_colors[c] for c in proc_cat['category']],
                    edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, proc_cat['attested_count']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'n={val}',
                ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('A. Phonological Processes by Domain\n(Total attested instances)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Attested Instances in Data', fontsize=11)
axes[0].set_ylim(0, 55)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Process frequency ranking
df_proc_sorted = df_proc.sort_values('attested_count', ascending=True)
colors_proc = [cat_colors[c] for c in df_proc_sorted['category']]
axes[1].barh(df_proc_sorted['process'], df_proc_sorted['attested_count'],
              color=colors_proc, edgecolor='white', linewidth=1, height=0.65)
axes[1].set_title('B. Ranked Phonological Process Frequency\n(Attested instances from field data)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xlabel('Attested Instances', fontsize=11)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')
legend_pats = [mpatches.Patch(color=v, label=k) for k,v in cat_colors.items()]
axes[1].legend(handles=legend_pats, fontsize=9, loc='lower right')

# Panel C: Direction of processes
dir_cat_counts = df_proc.groupby(['direction','category']).size().unstack(fill_value=0)
dir_cat_counts.plot(kind='bar', ax=axes[2], color=['#E63946','#457B9D','#1D3557'],
                    edgecolor='white', linewidth=1, width=0.6)
axes[2].set_title('C. Directionality of Phonological Processes\nby Domain', fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('Direction of Process', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].legend(title='Domain', fontsize=9)
axes[2].spines[['top','right']].set_visible(False)
axes[2].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig4_phono_processes.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 4")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5: Hierarchical Clustering of Consonants
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('white')

# Dendrogram
X_scaled_cons = StandardScaler().fit_transform(X)
Z = linkage(X_scaled_cons, method='ward')
dend = dendrogram(Z, labels=df_cons['phoneme'].tolist(), ax=axes[0],
                  color_threshold=3.5, above_threshold_color='gray',
                  leaf_font_size=13, leaf_rotation=0)
axes[0].set_title('A. Hierarchical Clustering of Kohomono Consonants\n(Ward linkage, distinctive feature distance)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Euclidean Distance (standardized)', fontsize=11)
axes[0].set_xlabel('Consonant Phonemes', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Similarity heatmap
dist_matrix = squareform(pdist(X_scaled_cons, metric='euclidean'))
sim_matrix = 1 / (1 + dist_matrix)
sns.heatmap(sim_matrix, ax=axes[1], cmap='YlOrRd',
            xticklabels=df_cons['phoneme'], yticklabels=df_cons['phoneme'],
            linewidths=0.3, linecolor='white', cbar_kws={'label': 'Phonological Similarity'})
axes[1].set_title('B. Phonological Similarity Matrix\n(Feature-based pairwise similarity)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xticklabels(df_cons['phoneme'], fontsize=9)
axes[1].set_yticklabels(df_cons['phoneme'], fontsize=9)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig5_clustering.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 5")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 6: Tonal System Analysis
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('white')
axes = axes.flatten()

# Panel A: Tone contrast types
contrast_counts = df_tone['contrast_type'].value_counts()
colors_tone = ['#E63946','#457B9D','#F4A261']
bars_t = axes[0].bar(contrast_counts.index, contrast_counts.values,
                      color=colors_tone[:len(contrast_counts)], edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars_t, contrast_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'n={val}',
                ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('A. Types of Tonal Contrast in Kohomono\n(Minimal pair analysis)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Minimal Pairs', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Tone level difference distribution
tone_diff_counts = df_tone['tone_level_diff'].value_counts().sort_index()
axes[1].bar(['Same Tone Level\n(Homophony)','Different Tone Level\n(True Contrast)'],
             [tone_diff_counts.get(0,0), tone_diff_counts.get(1,0)],
             color=['#F4A261','#1D3557'], edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(axes[1].patches, [tone_diff_counts.get(0,0), tone_diff_counts.get(1,0)]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'n={val}',
                ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('B. Tonal Homophony vs Lexical Contrast\n(Word pairs with identical phoneme sequences)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_ylim(0, 12)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')

# Panel C: Simulated F0 contours for the two basic tones
t = np.linspace(0, 1, 100)
# High tone - steady high
f0_high = 180 + 5*np.sin(2*np.pi*t*0.5) + np.random.normal(0, 2, 100)
# Low tone - steady low  
f0_low = 120 + 3*np.sin(2*np.pi*t*0.5) + np.random.normal(0, 2, 100)
# Rising contour tone
f0_rising = 115 + 70*t + np.random.normal(0, 2, 100)
# Falling contour tone
f0_falling = 185 - 65*t + np.random.normal(0, 2, 100)
axes[2].plot(t, f0_high, color='#E63946', lw=2.5, label='High tone (H) ˊ')
axes[2].plot(t, f0_low, color='#1D3557', lw=2.5, label='Low tone (L) ˋ')
axes[2].plot(t, f0_rising, color='#2A9D8F', lw=2.5, linestyle='--', label='Rising tone (LH) ˇ')
axes[2].plot(t, f0_falling, color='#F4A261', lw=2.5, linestyle='--', label='Falling tone (HL) ˆ')
axes[2].set_title('C. Schematic F0 Contours: Kohomono Tone System\n(H, L, LH, HL — 4 functional tone categories)', fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('Time (normalized)', fontsize=11)
axes[2].set_ylabel('F0 (Hz)', fontsize=11)
axes[2].legend(fontsize=9, loc='center right')
axes[2].set_facecolor('#F8F9FA')
axes[2].spines[['top','right']].set_visible(False)

# Panel D: Tonal process types from Autosegmental analysis
tonal_procs = ['Tonal Spreading','Tonal Delinking','Tonal Relinking','Downstep']
tonal_counts = [11, 8, 8, 5]
tonal_colors = ['#E63946','#457B9D','#F4A261','#2A9D8F']
bars_tp = axes[3].barh(tonal_procs, tonal_counts, color=tonal_colors, edgecolor='white', linewidth=1.2, height=0.5)
for bar, val in zip(bars_tp, tonal_counts):
    axes[3].text(bar.get_width()+0.15, bar.get_y()+bar.get_height()/2, f'n={val}',
                va='center', fontsize=11, fontweight='bold')
axes[3].set_title('D. Tonal Process Frequency\n(Autosegmental analysis, attested instances)', fontsize=11, fontweight='bold', pad=8)
axes[3].set_xlabel('Attested Instances', fontsize=11)
axes[3].set_xlim(0, 15)
axes[3].spines[['top','right']].set_visible(False)
axes[3].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig6_tonal_analysis.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 6")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 7: Syllable Structure Analysis
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

# Panel A: Syllable type distribution
syll_colors = ['#1D3557','#457B9D','#E63946','#F4A261','#2A9D8F','#A8DADC','#264653','#E9C46A']
bars_s = axes[0].bar(df_syll['structure'], df_syll['count'],
                      color=syll_colors[:len(df_syll)], edgecolor='white', linewidth=1.2, width=0.6)
for bar, val in zip(bars_s, df_syll['count']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(val),
                ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('A. Syllable Structure Frequency\n(Count per type in corpus)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Word Count', fontsize=11)
axes[0].set_xlabel('Syllable Structure', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Open vs Closed syllables
open_closed = df_syll.groupby('open_closed')['count'].sum()
axes[1].pie(open_closed.values, labels=[l.capitalize() for l in open_closed.index],
             colors=['#1D3557','#E63946','#F4A261'], autopct='%1.1f%%', startangle=140,
             wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':11})
axes[1].set_title('B. Open vs Closed Syllable Distribution\n(Kohomono syllable types)', fontsize=11, fontweight='bold', pad=8)

# Panel C: Word class by syllable structure
class_colors = {'noun':'#1D3557','verb':'#E63946','pronoun':'#F4A261',
                'prefix':'#2A9D8F','sentence':'#A8DADC'}
for i, row in df_syll.iterrows():
    col = class_colors.get(row['word_class'],'gray')
    axes[2].barh(row['structure'], row['count'], color=col, edgecolor='white', linewidth=1, height=0.6)
axes[2].set_title('C. Syllable Structures by Word Class', fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('Word Count', fontsize=11)
legend_pats = [mpatches.Patch(color=v, label=k.capitalize()) for k,v in class_colors.items()]
axes[2].legend(handles=legend_pats, fontsize=9, loc='lower right')
axes[2].spines[['top','right']].set_visible(False)
axes[2].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig7_syllable_structure.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 7")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 8: Comparative Typological Analysis
# ═══════════════════════════════════════════════════════════════════════════════
# Compare Kohomono phoneme inventory with related Upper Cross languages
lang_compare = {
    'Language': ['Kohomono\n(this study)','Lokaa\n(Iwara 1989)','Leggbo\n(Udoh 2014)',
                 'Mbembe\n(Akpan 2007)','Efik\n(review)','Ibibio\n(Essien)','Anaang\n(Udoh 2012)'],
    'Consonants': [24, 21, 23, 26, 20, 18, 20],
    'Vowels':     [10, 8,  7,  7,  6,  7,  7],
    'Tone_levels':[4,  2,  3,  3,  5,  3,  3],
    'Nasal_cons': [4,  3,  5,  3,  4,  4,  4],
    'Family':     ['Upper Cross','Upper Cross','Upper Cross','Upper Cross','Lower Cross','Lower Cross','Lower Cross'],
}
df_lang = pd.DataFrame(lang_compare)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
fig.patch.set_facecolor('white')

family_colors = {'Upper Cross':'#1D3557','Lower Cross':'#E63946'}
bar_fam_colors = [family_colors[f] for f in df_lang['Family']]

for ax, metric, title in zip(axes, 
    ['Consonants','Vowels','Tone_levels','Nasal_cons'],
    ['A. Consonant Phoneme Inventory Size','B. Vowel Phoneme Inventory Size',
     'C. Number of Functional Tone Levels','D. Nasal Consonant Count']):
    bars = ax.bar(range(len(df_lang)), df_lang[metric], color=bar_fam_colors,
                   edgecolor='white', linewidth=1.2, width=0.65)
    ax.set_xticks(range(len(df_lang)))
    ax.set_xticklabels(df_lang['Language'], fontsize=9)
    for bar, val in zip(bars, df_lang[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, str(val),
               ha='center', fontsize=10, fontweight='bold')
    # Highlight Kohomono
    bars[0].set_edgecolor('#F4A261')
    bars[0].set_linewidth(3)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel('Count', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('#F8F9FA')
    legend_els = [mpatches.Patch(color=v, label=k) for k,v in family_colors.items()]
    ax.legend(handles=legend_els, fontsize=9)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig8_typological_comparison.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 8")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 9: Distinctive Feature Correlation and Confusion Matrix
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

# Panel A: Feature intercorrelation
feat_df_plot = df_cons[['voiced','labial','coronal','dorsal','sonorant','continuant','nasal_feat','word_initial','word_final']]
feat_df_plot.columns = ['Voiced','Labial','Coronal','Dorsal','Sonorant','Continuant','Nasal','Initial','Final']
corr = feat_df_plot.corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Pearson r', 'shrink':0.8}, annot_kws={'size':9})
axes[0].set_title('A. Distinctive Feature Intercorrelations\n(Pearson r, lower triangle)', fontsize=11, fontweight='bold', pad=8)

# Panel B: Confusion matrix of RF classifier (using augmented data)
from sklearn.model_selection import cross_val_predict
np.random.seed(42)
X_aug2 = np.vstack([X + np.random.normal(0, 0.1, X.shape) for _ in range(15)])
y_aug2 = np.tile(y_enc, 15)
rf_pred = cross_val_predict(RandomForestClassifier(n_estimators=100, random_state=42),
                             X_aug2, y_aug2, cv=4)
cm = confusion_matrix(y_aug2, rf_pred)
class_names = le.classes_
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            xticklabels=[c.capitalize() for c in class_names],
            yticklabels=[c.capitalize() for c in class_names],
            linewidths=0.5, linecolor='white', cbar_kws={'label':'Proportion'})
axes[1].set_title('B. Random Forest Confusion Matrix\nConsonant Manner Classification (Normalized)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xlabel('Predicted Manner', fontsize=11)
axes[1].set_ylabel('True Manner', fontsize=11)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig9_feature_corr_confusion.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 9")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 10: ATR Vowel Harmony Computational Model
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel A: ATR groupings
atr_plus = df_vowels[df_vowels['ATR']==1]
atr_minus = df_vowels[df_vowels['ATR']==0]
vowel_positions = {
    'i':(1,5),'e':(1.5,3.5),'u':(5,5),'o':(4.5,3.5),'o:':(4.7,3.7),
    'a':(3,1),'a:':(3.2,1.2),'ɛ':(1.8,2),'ɔ':(4.2,2),'ɔ:':(4.4,2.2)
}
# ATR+ zone
circle1 = plt.Circle((2.8,3.5), 2.5, color='#1D3557', fill=True, alpha=0.12, linewidth=2)
circle2 = plt.Circle((3.2,3.0), 2.8, color='#E63946', fill=True, alpha=0.10, linewidth=2, linestyle='--')
axes[0].add_patch(circle1); axes[0].add_patch(circle2)
for v, (x, y) in vowel_positions.items():
    row = df_vowels[df_vowels['phoneme']==v]
    if len(row) > 0:
        is_atr = row['ATR'].values[0]
        col = '#1D3557' if is_atr else '#E63946'
        ls = '-' if ':' not in v else '--'
        axes[0].scatter(x, y, s=350, color=col, zorder=5, edgecolors='white', linewidth=2)
        axes[0].text(x, y+0.35, f'/{v}/', ha='center', fontsize=12, fontweight='bold', color=col)
axes[0].set_xlim(0,6); axes[0].set_ylim(0,6.5)
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title('A. ATR Vowel Harmony Sets in Kohomono\n([+ATR] blue; [-ATR] red)', fontsize=11, fontweight='bold', pad=8)
legend_els = [mpatches.Patch(color='#1D3557', label='+ATR set: /i, e, u, o, o:/'),
              mpatches.Patch(color='#E63946', label='-ATR set: /ɛ, a, a:, ɔ, ɔ:/')]
axes[0].legend(handles=legend_els, fontsize=10, loc='lower center')
axes[0].set_facecolor('#F8F9FA')

# Panel B: Vowel feature profile heatmap
vowel_feat_cols = ['ATR','high_feat','low_feat','back_feat','long_feat']
vowel_feat_labels = ['ATR','+High','+Low','+Back','+Long']
vowel_matrix = df_vowels[vowel_feat_cols].values.T
sns.heatmap(vowel_matrix, ax=axes[1], cmap='RdYlBu_r',
            xticklabels=df_vowels['phoneme'], yticklabels=vowel_feat_labels,
            linewidths=0.8, linecolor='white', cbar_kws={'label':'Feature Value'},
            annot=True, fmt='d', annot_kws={'size':11})
axes[1].set_title('B. Vowel Distinctive Feature Matrix\n(10 phonemic vowels × 5 features)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xticklabels(df_vowels['phoneme'], fontsize=12)
axes[1].set_yticklabels(vowel_feat_labels, fontsize=10, rotation=0)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig10_vowel_harmony.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 10")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 11: Comprehensive Metrics Summary Heatmap
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('white')
summary_data = {
    'Model': ['Random Forest','Logistic Reg.','SVM (RBF)','k-NN','Grad. Boost'],
    'CV Mean': [cv_results[n].mean() for n in models],
    'CV Std':  [cv_results[n].std()  for n in models],
    'Min Fold':[cv_results[n].min()  for n in models],
    'Max Fold':[cv_results[n].max()  for n in models],
}
summary_df = pd.DataFrame(summary_data).set_index('Model')
summary_num = summary_df.copy()
sns.heatmap(summary_num, annot=summary_df.round(3).values, fmt='', cmap='YlOrRd',
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Accuracy Score'}, annot_kws={'size':12, 'weight':'bold'})

ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, rotation=0)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig11_metrics_summary.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 11")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 12: Language Endangerment & Documentation Impact
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel A: Speaker population decline simulation
years = np.array([1963, 1973, 1991, 2006, 2019, 2024, 2030, 2040])
speakers = np.array([55000, 48000, 38000, 35000, 33000, 31000, 27000, 22000])
is_projected = np.array([0,0,0,0,0,0,1,1], dtype=bool)
axes[0].plot(years[~is_projected], speakers[~is_projected], 'o-', color='#1D3557', lw=2.5, 
             markersize=8, label='Documented (census/Ethnologue)')
axes[0].plot(years[is_projected], speakers[is_projected], 'o--', color='#E63946', lw=2.5,
             markersize=8, label='Projected (without intervention)')
axes[0].axvline(2020, color='#F4A261', linestyle=':', lw=2, label='This study (Etu 2020)')
axes[0].fill_between(years, speakers*0.85, speakers*1.15, alpha=0.1, color='#1D3557')
axes[0].set_title('A. Kohomono Speaker Population Trend\n(Documented decline; ~33,000 speakers in 2019)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_xlabel('Year', fontsize=11); axes[0].set_ylabel('Estimated Speakers', fontsize=11)
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_facecolor('#F8F9FA'); axes[0].spines[['top','right']].set_visible(False)

# Panel B: Documentation coverage of sound system
categories = ['Phonemic\nConsonants','Vowel\nPhonemes','Tone\nTypes','Phono.\nProcesses','Syllable\nStructures','Allophones\nDocumented']
this_study = [24, 10, 4, 13, 8, 18]
cook_1969  = [23, 5, 0, 0, 0, 5]
njoku_2016 = [31, 11, 2, 3, 2, 8]
x = np.arange(len(categories))
width = 0.28
axes[1].bar(x - width, cook_1969, width, label='Cook (1969)', color='#A8DADC', edgecolor='white')
axes[1].bar(x, njoku_2016, width, label='Njoku (2016)', color='#457B9D', edgecolor='white')
axes[1].bar(x + width, this_study, width, label='Etu (2020) — This Study', color='#1D3557', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(categories, fontsize=9)
axes[1].set_title('B. Cumulative Documentation of Kohomono\nPhonological System Across Studies', fontsize=11, fontweight='bold', pad=8)
axes[1].set_ylabel('Units Documented', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig12_endangerment_documentation.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 12")

print(f"\n✅ All 12 figures saved to {FIGDIR}/")
print("\n=== ML RESULTS SUMMARY ===")
for name, short in zip(models.keys(), model_names_short):
    scores = cv_results[name]
    print(f"{name:20s} | CV Mean={scores.mean():.3f} | SD={scores.std():.3f} | Range=[{scores.min():.3f}-{scores.max():.3f}]")


Data loaded. Consonants: 24 | Vowels: 10
Processes: 13 | Tone pairs: 15
Saved Fig 1
Saved Fig 2
Saved Fig 3
Saved Fig 4
Saved Fig 5
Saved Fig 6
Saved Fig 7
Saved Fig 8
Saved Fig 9
Saved Fig 10
Saved Fig 11
Saved Fig 12

✅ All 12 figures saved to kohomono_figures/

=== ML RESULTS SUMMARY ===
Random Forest        | CV Mean=0.917 | SD=0.083 | Range=[0.833-1.000]
Logistic Reg.        | CV Mean=0.833 | SD=0.118 | Range=[0.667-1.000]
SVM (RBF)            | CV Mean=0.792 | SD=0.072 | Range=[0.667-0.833]
k-NN                 | CV Mean=0.583 | SD=0.186 | Range=[0.333-0.833]
Grad. Boost          | CV Mean=0.917 | SD=0.083 | Range=[0.833-1.000]


# CODE TWO

In [11]:
"""
Machine Learning Analysis of Kohomono Phonology Data
PhD thesis: "A Phonology of Kohomono" by Etu, Mercy Runyi
University of Port Harcourt
Manuscript for Language Sciences (Elsevier) - Sociolinguistics and AI Special Issue
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
DPI = 600
FIGDIR = "kohomono_figures2"
import os; os.makedirs(FIGDIR, exist_ok=True)

# ─── KOHOMONO PHONOLOGICAL DATA ────────────────────────────────────────────────
# Based on Etu (2020) primary field data from Kohomono language, Cross River State

# Consonant inventory with phonological features
consonants_data = {
    'phoneme': ['p','b','t','d','k','g','kw','gw','kp','gb','dʒ',
                'm','n','ɲ','ŋ','f','v','s','z','ʃ','h','ɹ','j','w'],
    'voiced':  [0,  1,  0,  1,  0,  1,  0,   1,  0,  1,  1,
                1,  1,  1,  1,  0,  1,  0,  1,  0,  0,  1,  1,  1],
    'manner':  ['stop','stop','stop','stop','stop','stop','stop','stop','stop','stop','affricate',
                'nasal','nasal','nasal','nasal','fricative','fricative','fricative','fricative','fricative','fricative','approximant','approximant','approximant'],
    'place':   ['bilabial','bilabial','alveolar','alveolar','velar','velar','labiovelar','labiovelar','labio-velar','labio-velar','palatal',
                'bilabial','alveolar','palatal','velar','labiodental','labiodental','alveolar','alveolar','palato-alveolar','glottal','alveolar','palatal','bilabial'],
    'labial':  [1,1,0,0,0,0,1,1,1,1,0,  1,0,0,0,1,1,0,0,0,0,0,0,1],
    'coronal': [0,0,1,1,0,0,0,0,0,0,1,  0,1,1,0,0,0,1,1,1,0,1,1,0],
    'dorsal':  [0,0,0,0,1,1,1,1,1,1,0,  0,0,0,1,0,0,0,0,0,0,0,0,0],
    'sonorant':[0,0,0,0,0,0,0,0,0,0,0,  1,1,1,1,0,0,0,0,0,0,1,1,1],
    'continuant':[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1],
    'nasal_feat':[0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0],
    'word_initial':[1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,0,0,1,1],
    'word_medial': [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    'word_final':  [1,1,1,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0],
    'has_allophones':[1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0],
    'freq_score':  [3,5,4,3,4,2,3,3,4,3,2,  5,5,4,3,4,3,4,3,2,3,4,3,3],
}

df_cons = pd.DataFrame(consonants_data)

# Vowel inventory with features
vowels_data = {
    'phoneme': ['i','e','ɛ','a','a:','u','o','o:','ɔ','ɔ:'],
    'height':  ['high','mid','mid-low','low','low','high','mid','mid','mid-low','mid-low'],
    'backness':['front','front','front','central','central','back','back','back','back','back'],
    'length':  ['short','short','short','short','long','short','short','long','short','long'],
    'ATR':     [1,1,0,0,0,1,1,1,0,0],
    'high_feat':[1,0,0,0,0,1,0,0,0,0],
    'low_feat': [0,0,0,1,1,0,0,0,0,0],
    'back_feat':[0,0,0,0,0,1,1,1,1,1],
    'long_feat':[0,0,0,0,1,0,0,1,0,1],
    'freq_score':[5,4,4,5,2,4,4,2,4,2],
}
df_vowels = pd.DataFrame(vowels_data)

# Tone data - lexical minimal pairs showing tonal contrast
tone_data = {
    'word_pair': ['kpè/kpé','gwá/gwá','ìʃín/ìʃín','kpé/kpé','bàn/bàn','jén/jém',
                  'ézèm/ézèn','tàn/dàn','bà/pà','kù/ɡú','àkà/áɡà','ókó/óɡó',
                  'úpà/úkpà','ótà/ódà','ótàm/ódàm'],
    'tone1': ['L','H','LH','H','L','H','LH','L','L','L','LL','HH','HH','HH','HHH'],
    'tone2': ['H','H','LH','H','L','H','LH','L','L','H','HH','HH','HH','HH','HHH'],
    'meaning1': ['kneel','drink','hair','teach','marry','eye','crocodile','sell','women','in','mother','vegetable','calabash','string','traveller'],
    'meaning2': ['learn','cover','grass','learn/pay','smooth','take','meat','smooth','arrive','smoked','thread','in-law','valley','admit','be good'],
    'contrast_type': ['lexical','lexical','homophony','homophony','lexical','lexical','lexical','consonant','consonant','consonant','consonant','consonant','consonant','consonant','consonant'],
    'tone_level_diff': [1,0,0,0,0,0,0,0,0,1,1,0,1,1,1],
}
df_tone = pd.DataFrame(tone_data)

# Phonological processes data
processes_data = {
    'process': ['Consonant Assimilation','Gemination','Consonant Devoicing','Consonant Weakening',
                'Labialization','Palatalization','Nasalization','Vowel Harmony',
                'Vowel Deletion','Vowel Lengthening','Tonal Spreading','Tonal Delinking','Downstep'],
    'category': ['Consonant','Consonant','Consonant','Consonant',
                 'Consonant','Consonant','Vowel','Vowel',
                 'Vowel','Vowel','Tonal','Tonal','Tonal'],
    'direction': ['Progressive','Both','Regressive','Progressive',
                  'Progressive','Progressive','Both','Both',
                  'Regressive','Both','Both','Regressive','Progressive'],
    'frequency': [5,4,4,5,3,3,3,4,3,3,4,3,3],
    'environment': ['Word boundary','Word boundary','Final position','Intervocalic',
                    'Before back vowel','Before front vowel','Near nasals','Morpheme internal',
                    'Between vowels','Compensatory','Tone sequence','Vowel deletion','H-tone sequence'],
    'attested_count': [18,12,15,10,8,6,9,14,7,6,11,8,5],
}
df_proc = pd.DataFrame(processes_data)

# Syllable structure data
syllable_data = {
    'structure': ['V','CV','CVC','VCV','CVCV','CVVCV','N̩','N̩CV'],
    'count':     [12, 45, 38,  52,   87,   23,   8,    15],
    'examples':  ['ò (she)','sé (sit)','ʃìn (fifteen)','ónò (tears)','bùʃí (yam)','bóvén (thing)','N̩- (1sg)','N̩bà (I am back)'],
    'word_class':['pronoun','verb','noun','noun','noun','noun','prefix','sentence'],
    'open_closed':['open','open','closed','open','open','open','syllabic','closed'],
}
df_syll = pd.DataFrame(syllable_data)

print("Data loaded. Consonants:", len(df_cons), "| Vowels:", len(df_vowels))
print("Processes:", len(df_proc), "| Tone pairs:", len(df_tone))

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: Kohomono Consonant Inventory Overview
# ═══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('white')
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel A: Consonants by Manner
ax1 = fig.add_subplot(gs[0, 0])
manner_counts = df_cons['manner'].value_counts()
colors_manner = ['#1D3557','#457B9D','#A8DADC','#E63946','#F4A261','#2A9D8F']
bars = ax1.barh(manner_counts.index, manner_counts.values, color=colors_manner[:len(manner_counts)], 
                edgecolor='white', linewidth=1.2, height=0.65)
for bar, val in zip(bars, manner_counts.values):
    ax1.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'n={val}', 
             va='center', fontsize=10, fontweight='bold')
ax1.set_title('A. Consonants by Manner of Articulation', fontsize=11, fontweight='bold', pad=8)
ax1.set_xlabel('Count', fontsize=10)
ax1.set_xlim(0, 13)
ax1.spines[['top','right']].set_visible(False)
ax1.set_facecolor('#F8F9FA')

# Panel B: Consonants by Place
ax2 = fig.add_subplot(gs[0, 1])
place_counts = df_cons['place'].value_counts()
colors_place = ['#E63946','#457B9D','#F4A261','#2A9D8F','#264653','#A8DADC','#E9C46A']
bars2 = ax2.barh(place_counts.index, place_counts.values, color=colors_place[:len(place_counts)], 
                 edgecolor='white', linewidth=1.2, height=0.65)
for bar, val in zip(bars2, place_counts.values):
    ax2.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'n={val}', 
             va='center', fontsize=10, fontweight='bold')
ax2.set_title('B. Consonants by Place of Articulation', fontsize=11, fontweight='bold', pad=8)
ax2.set_xlabel('Count', fontsize=10)
ax2.set_xlim(0, 10)
ax2.spines[['top','right']].set_visible(False)
ax2.set_facecolor('#F8F9FA')

# Panel C: Voiced vs Voiceless
ax3 = fig.add_subplot(gs[0, 2])
voice_counts = df_cons['voiced'].value_counts()
labels = ['Voiced', 'Voiceless']
sizes = [voice_counts.get(1, 0), voice_counts.get(0, 0)]
wedge_colors = ['#E63946', '#457B9D']
wedges, texts, autotexts = ax3.pie(sizes, labels=labels, colors=wedge_colors, autopct='%1.1f%%',
                                    startangle=140, textprops={'fontsize': 11},
                                    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for at in autotexts:
    at.set_fontsize(12); at.set_fontweight('bold')
ax3.set_title('C. Voicing Distribution\n(24 consonant phonemes)', fontsize=11, fontweight='bold', pad=8)

# Panel D: Positional Distribution of Consonants
ax4 = fig.add_subplot(gs[1, 0])
positions = ['Word Initial', 'Word Medial', 'Word Final']
pos_counts = [df_cons['word_initial'].sum(), df_cons['word_medial'].sum(), df_cons['word_final'].sum()]
bar_pos = ax4.bar(positions, pos_counts, color=['#1D3557','#457B9D','#A8DADC'], 
                   edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bar_pos, pos_counts):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'n={val}', 
             ha='center', fontsize=11, fontweight='bold')
ax4.set_title('D. Consonant Positional Distribution', fontsize=11, fontweight='bold', pad=8)
ax4.set_ylabel('Number of Phonemes', fontsize=10)
ax4.set_ylim(0, 28)
ax4.spines[['top','right']].set_visible(False)
ax4.set_facecolor('#F8F9FA')

# Panel E: Vowel inventory positions
ax5 = fig.add_subplot(gs[1, 1])
vowel_plot = {
    'i': (0.1, 0.95), 'e': (0.15, 0.6), 'ɛ': (0.2, 0.35),
    'a': (0.5, 0.05), 'a:': (0.52, 0.05),
    'u': (0.9, 0.95), 'o': (0.85, 0.6), 'o:': (0.87, 0.6),
    'ɔ': (0.8, 0.35), 'ɔ:': (0.82, 0.35)
}
for v, (x, y) in vowel_plot.items():
    color = '#E63946' if ':' in v else '#1D3557'
    size = 180 if ':' in v else 220
    ax5.scatter(x, y, s=size, color=color, zorder=5, edgecolors='white', linewidth=1.5)
    ax5.text(x, y+0.055, f'/{v}/', ha='center', va='bottom', fontsize=11, 
             fontweight='bold', color=color)
# Draw vowel trapezoid outline
ax5.plot([0.05, 0.95], [1.0, 1.0], 'gray', linewidth=1, alpha=0.4, linestyle='--')
ax5.plot([0.05, 0.5],  [1.0, 0.0], 'gray', linewidth=1, alpha=0.4, linestyle='--')
ax5.plot([0.95, 0.5],  [1.0, 0.0], 'gray', linewidth=1, alpha=0.4, linestyle='--')
ax5.set_xlim(-0.05, 1.1); ax5.set_ylim(-0.1, 1.15)
ax5.set_xticks([0.05, 0.5, 0.95]); ax5.set_xticklabels(['Front','Central','Back'], fontsize=10)
ax5.set_yticks([0.0, 0.35, 0.6, 0.95]); ax5.set_yticklabels(['Low','Mid-Low','Mid','High'], fontsize=10)
ax5.set_title('E. Vowel Phoneme Space\n(7 short + 3 long = 10 vowel phonemes)', fontsize=11, fontweight='bold', pad=8)
legend_els = [mpatches.Patch(color='#1D3557', label='Short vowels (7)'),
              mpatches.Patch(color='#E63946', label='Long vowels (3)')]
ax5.legend(handles=legend_els, fontsize=9, loc='lower right')
ax5.set_facecolor('#F8F9FA')

# Panel F: Phonological feature heatmap
ax6 = fig.add_subplot(gs[1, 2])
feat_cols = ['voiced','labial','coronal','dorsal','sonorant','continuant','nasal_feat']
feat_labels = ['Voiced','Labial','Coronal','Dorsal','Sonorant','Continuant','Nasal']
feat_matrix = df_cons[feat_cols].values.T
sns.heatmap(feat_matrix, ax=ax6, cmap='Blues', xticklabels=df_cons['phoneme'],
            yticklabels=feat_labels, cbar=False, linewidths=0.3, linecolor='white',
            annot=False)
ax6.set_title('F. Distinctive Feature Matrix\n(24 consonant phonemes)', fontsize=11, fontweight='bold', pad=8)
ax6.set_xticklabels(df_cons['phoneme'], fontsize=8, rotation=0)
ax6.set_yticklabels(feat_labels, fontsize=9, rotation=0)


plt.savefig(f'{FIGDIR}/fig1_phoneme_inventory.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 1")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: Machine Learning Phoneme Classification
# ═══════════════════════════════════════════════════════════════════════════════
# Classify consonants into manner classes using distinctive features
feature_cols = ['voiced','labial','coronal','dorsal','sonorant','continuant','nasal_feat','word_initial','word_medial','word_final']
X = df_cons[feature_cols].values
y_manner = df_cons['manner'].values
le = LabelEncoder()
y_enc = le.fit_transform(y_manner)

# Use all data with cross-validation (small dataset)
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Reg.': LogisticRegression(max_iter=1000, random_state=42, C=1.0),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42, probability=True),
    'k-NN': KNeighborsClassifier(n_neighbors=3),
    'Grad. Boost': GradientBoostingClassifier(n_estimators=80, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

model_names_short = ['RF','LR','SVM','k-NN','GB']
bar_colors = ['#1D3557','#457B9D','#E63946','#F4A261','#2A9D8F']
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y_enc, cv=skf, scoring='accuracy')
    cv_results[name] = scores

# Panel A: CV Accuracy
means = [cv_results[n].mean() for n in models]
stds  = [cv_results[n].std()  for n in models]
bars = axes[0].bar(model_names_short, means, yerr=stds, color=bar_colors, 
                    capsize=6, edgecolor='white', linewidth=1.2, width=0.6,
                    error_kw={'elinewidth':2,'ecolor':'#333'})
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('A. Consonant Class Classification\n(4-Fold CV Accuracy ± SD)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_ylim(0, 1.15)
axes[0].axhline(0.8, color='crimson', linestyle='--', alpha=0.5, linewidth=1.5, label='0.80 baseline')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Feature importance from RF
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X, y_enc)
feat_imp = rf_model.feature_importances_
feat_labels_short = ['Voiced','Labial','Coronal','Dorsal','Sonorant','Continuant','Nasal','Init.','Med.','Final']
sorted_idx = np.argsort(feat_imp)[::-1]
axes[1].barh([feat_labels_short[i] for i in sorted_idx], feat_imp[sorted_idx],
              color=['#E63946' if i < 3 else '#457B9D' for i in range(len(sorted_idx))],
              edgecolor='white', linewidth=1, height=0.6)
axes[1].set_title('B. Feature Importance\n(Random Forest, Consonant Classification)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xlabel('Importance Score', fontsize=11)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')

# Panel C: PCA of consonant feature space
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

manner_colors = {'stop':'#1D3557','nasal':'#E63946','fricative':'#F4A261',
                 'approximant':'#2A9D8F','affricate':'#A8DADC'}
for manner in df_cons['manner'].unique():
    mask = df_cons['manner'] == manner
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=manner_colors.get(manner,'gray'), s=120, label=manner.capitalize(),
                   edgecolors='white', linewidth=1.5, zorder=5, alpha=0.9)
for i, row in df_cons.iterrows():
    axes[2].annotate(f'/{row["phoneme"]}/', (X_pca[i,0], X_pca[i,1]),
                    textcoords='offset points', xytext=(4,4), fontsize=7.5, alpha=0.85)
axes[2].set_title(f'C. PCA of Consonant Feature Space\n(PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%})',
                  fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('PC1', fontsize=11); axes[2].set_ylabel('PC2', fontsize=11)
axes[2].legend(fontsize=8, loc='lower right', framealpha=0.9)
axes[2].spines[['top','right']].set_visible(False)
axes[2].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig2_ml_classification.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 2")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: Cross-Validation and Model Performance
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel A: Violin plot of CV scores
fold_data = []
for name, short in zip(models.keys(), model_names_short):
    for score in cv_results[name]:
        fold_data.append({'Model': short, 'Accuracy': score})
fold_df = pd.DataFrame(fold_data)
sns.violinplot(data=fold_df, x='Model', y='Accuracy', palette=bar_colors, ax=axes[0],
               inner='point', linewidth=1.5)
axes[0].set_title('A. Cross-Validation Score Distributions\n(4-fold stratified CV)', fontsize=12, fontweight='bold', pad=8)
axes[0].set_ylabel('Classification Accuracy', fontsize=11)
axes[0].set_xlabel('Classifier', fontsize=11)
axes[0].axhline(0.8, color='crimson', linestyle='--', alpha=0.5, linewidth=1.5)
axes[0].set_ylim(0.3, 1.15)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Learning curves for best model (RF)
from sklearn.model_selection import learning_curve as lc_fn
# Simulate learning curve with extended synthetic augmented data
np.random.seed(99)
X_aug = np.vstack([X + np.random.normal(0, 0.08, X.shape) for _ in range(12)])
y_aug = np.tile(y_enc, 12)
train_sizes, train_scores, val_scores = lc_fn(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_aug, y_aug, cv=4, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1)
axes[1].plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#1D3557', lw=2.5, label='Training')
axes[1].fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1),
                      train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.15, color='#1D3557')
axes[1].plot(train_sizes, val_scores.mean(axis=1), 's-', color='#E63946', lw=2.5, label='Validation')
axes[1].fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1),
                      val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.15, color='#E63946')
axes[1].set_title('B. Learning Curve (Random Forest)\nConsonant Manner Classification', fontsize=12, fontweight='bold', pad=8)
axes[1].set_xlabel('Training Samples', fontsize=11); axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].legend(fontsize=10); axes[1].set_ylim(0.4, 1.05)
axes[1].grid(True, alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig3_cv_learning.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 3")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: Phonological Processes Analysis
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

# Panel A: Processes by category
cat_colors = {'Consonant':'#1D3557','Vowel':'#457B9D','Tonal':'#E63946'}
proc_cat = df_proc.groupby('category')['attested_count'].sum().reset_index()
bars = axes[0].bar(proc_cat['category'], proc_cat['attested_count'],
                    color=[cat_colors[c] for c in proc_cat['category']],
                    edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, proc_cat['attested_count']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'n={val}',
                ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('A. Phonological Processes by Domain\n(Total attested instances)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Attested Instances in Data', fontsize=11)
axes[0].set_ylim(0, 55)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Process frequency ranking
df_proc_sorted = df_proc.sort_values('attested_count', ascending=True)
colors_proc = [cat_colors[c] for c in df_proc_sorted['category']]
axes[1].barh(df_proc_sorted['process'], df_proc_sorted['attested_count'],
              color=colors_proc, edgecolor='white', linewidth=1, height=0.65)
axes[1].set_title('B. Ranked Phonological Process Frequency\n(Attested instances from field data)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xlabel('Attested Instances', fontsize=11)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')
legend_pats = [mpatches.Patch(color=v, label=k) for k,v in cat_colors.items()]
axes[1].legend(handles=legend_pats, fontsize=9, loc='lower right')

# Panel C: Direction of processes
dir_cat_counts = df_proc.groupby(['direction','category']).size().unstack(fill_value=0)
dir_cat_counts.plot(kind='bar', ax=axes[2], color=['#E63946','#457B9D','#1D3557'],
                    edgecolor='white', linewidth=1, width=0.6)
axes[2].set_title('C. Directionality of Phonological Processes\nby Domain', fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('Direction of Process', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].legend(title='Domain', fontsize=9)
axes[2].spines[['top','right']].set_visible(False)
axes[2].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig4_phono_processes.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 4")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5: Hierarchical Clustering of Consonants
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('white')

# Dendrogram
X_scaled_cons = StandardScaler().fit_transform(X)
Z = linkage(X_scaled_cons, method='ward')
dend = dendrogram(Z, labels=df_cons['phoneme'].tolist(), ax=axes[0],
                  color_threshold=3.5, above_threshold_color='gray',
                  leaf_font_size=13, leaf_rotation=0)
axes[0].set_title('A. Hierarchical Clustering of Kohomono Consonants\n(Ward linkage, distinctive feature distance)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Euclidean Distance (standardized)', fontsize=11)
axes[0].set_xlabel('Consonant Phonemes', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Similarity heatmap
dist_matrix = squareform(pdist(X_scaled_cons, metric='euclidean'))
sim_matrix = 1 / (1 + dist_matrix)

sns.heatmap(
    sim_matrix,
    ax=axes[1],
    cmap='YlOrRd',
    annot=False,
    xticklabels=df_cons['phoneme'],
    yticklabels=df_cons['phoneme'],
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label':'Phonological Similarity'}
)

# Manual annotation inside cells
for i in range(sim_matrix.shape[0]):
    for j in range(sim_matrix.shape[1]):
        axes[1].text(
            j + 0.5,
            i + 0.5,
            f'{sim_matrix[i,j]:.2f}',
            ha='center',
            va='center',
            fontsize=5,
            fontweight='bold',
            color='black'
        )

axes[1].set_title(
'B. Phonological Similarity Matrix\n(Feature-based pairwise similarity)',
fontsize=11,
fontweight='bold',
pad=8
)

axes[1].set_xticklabels(
    df_cons['phoneme'],
    fontsize=9,
    rotation=0
)

axes[1].set_yticklabels(
    df_cons['phoneme'],
    fontsize=9,
    rotation=0
)

plt.tight_layout()
plt.savefig(
    f'{FIGDIR}/fig5_clustering.png',
    dpi=DPI,
    bbox_inches='tight',
    facecolor='white'
)

plt.close()
print("Saved Fig 5")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 6: Tonal System Analysis
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('white')
axes = axes.flatten()

# Panel A: Tone contrast types
contrast_counts = df_tone['contrast_type'].value_counts()
colors_tone = ['#E63946','#457B9D','#F4A261']
bars_t = axes[0].bar(contrast_counts.index, contrast_counts.values,
                      color=colors_tone[:len(contrast_counts)], edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars_t, contrast_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'n={val}',
                ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('A. Types of Tonal Contrast in Kohomono\n(Minimal pair analysis)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Minimal Pairs', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Tone level difference distribution
tone_diff_counts = df_tone['tone_level_diff'].value_counts().sort_index()
axes[1].bar(['Same Tone Level\n(Homophony)','Different Tone Level\n(True Contrast)'],
             [tone_diff_counts.get(0,0), tone_diff_counts.get(1,0)],
             color=['#F4A261','#1D3557'], edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(axes[1].patches, [tone_diff_counts.get(0,0), tone_diff_counts.get(1,0)]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'n={val}',
                ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('B. Tonal Homophony vs Lexical Contrast\n(Word pairs with identical phoneme sequences)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_ylim(0, 12)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')

# Panel C: Simulated F0 contours for the two basic tones
t = np.linspace(0, 1, 100)
# High tone - steady high
f0_high = 180 + 5*np.sin(2*np.pi*t*0.5) + np.random.normal(0, 2, 100)
# Low tone - steady low  
f0_low = 120 + 3*np.sin(2*np.pi*t*0.5) + np.random.normal(0, 2, 100)
# Rising contour tone
f0_rising = 115 + 70*t + np.random.normal(0, 2, 100)
# Falling contour tone
f0_falling = 185 - 65*t + np.random.normal(0, 2, 100)
axes[2].plot(t, f0_high, color='#E63946', lw=2.5, label='High tone (H) ˊ')
axes[2].plot(t, f0_low, color='#1D3557', lw=2.5, label='Low tone (L) ˋ')
axes[2].plot(t, f0_rising, color='#2A9D8F', lw=2.5, linestyle='--', label='Rising tone (LH) ˇ')
axes[2].plot(t, f0_falling, color='#F4A261', lw=2.5, linestyle='--', label='Falling tone (HL) ˆ')
axes[2].set_title('C. Schematic F0 Contours: Kohomono Tone System\n(H, L, LH, HL — 4 functional tone categories)', fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('Time (normalized)', fontsize=11)
axes[2].set_ylabel('F0 (Hz)', fontsize=11)
axes[2].legend(fontsize=9, loc='center right')
axes[2].set_facecolor('#F8F9FA')
axes[2].spines[['top','right']].set_visible(False)

# Panel D: Tonal process types from Autosegmental analysis
tonal_procs = ['Tonal Spreading','Tonal Delinking','Tonal Relinking','Downstep']
tonal_counts = [11, 8, 8, 5]
tonal_colors = ['#E63946','#457B9D','#F4A261','#2A9D8F']
bars_tp = axes[3].barh(tonal_procs, tonal_counts, color=tonal_colors, edgecolor='white', linewidth=1.2, height=0.5)
for bar, val in zip(bars_tp, tonal_counts):
    axes[3].text(bar.get_width()+0.15, bar.get_y()+bar.get_height()/2, f'n={val}',
                va='center', fontsize=11, fontweight='bold')
axes[3].set_title('D. Tonal Process Frequency\n(Autosegmental analysis, attested instances)', fontsize=11, fontweight='bold', pad=8)
axes[3].set_xlabel('Attested Instances', fontsize=11)
axes[3].set_xlim(0, 15)
axes[3].spines[['top','right']].set_visible(False)
axes[3].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig6_tonal_analysis.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 6")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 7: Syllable Structure Analysis
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('white')

# Panel A: Syllable type distribution
syll_colors = ['#1D3557','#457B9D','#E63946','#F4A261','#2A9D8F','#A8DADC','#264653','#E9C46A']
bars_s = axes[0].bar(df_syll['structure'], df_syll['count'],
                      color=syll_colors[:len(df_syll)], edgecolor='white', linewidth=1.2, width=0.6)
for bar, val in zip(bars_s, df_syll['count']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, str(val),
                ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('A. Syllable Structure Frequency\n(Count per type in corpus)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_ylabel('Word Count', fontsize=11)
axes[0].set_xlabel('Syllable Structure', fontsize=11)
axes[0].spines[['top','right']].set_visible(False)
axes[0].set_facecolor('#F8F9FA')

# Panel B: Open vs Closed syllables
open_closed = df_syll.groupby('open_closed')['count'].sum()
axes[1].pie(open_closed.values, labels=[l.capitalize() for l in open_closed.index],
             colors=['#1D3557','#E63946','#F4A261'], autopct='%1.1f%%', startangle=140,
             wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':11})
axes[1].set_title('B. Open vs Closed Syllable Distribution\n(Kohomono syllable types)', fontsize=11, fontweight='bold', pad=8)

# Panel C: Word class by syllable structure
class_colors = {'noun':'#1D3557','verb':'#E63946','pronoun':'#F4A261',
                'prefix':'#2A9D8F','sentence':'#A8DADC'}
for i, row in df_syll.iterrows():
    col = class_colors.get(row['word_class'],'gray')
    axes[2].barh(row['structure'], row['count'], color=col, edgecolor='white', linewidth=1, height=0.6)
axes[2].set_title('C. Syllable Structures by Word Class', fontsize=11, fontweight='bold', pad=8)
axes[2].set_xlabel('Word Count', fontsize=11)
legend_pats = [mpatches.Patch(color=v, label=k.capitalize()) for k,v in class_colors.items()]
axes[2].legend(handles=legend_pats, fontsize=9, loc='lower right')
axes[2].spines[['top','right']].set_visible(False)
axes[2].set_facecolor('#F8F9FA')


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig7_syllable_structure.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 7")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 8: Comparative Typological Analysis
# ═══════════════════════════════════════════════════════════════════════════════
# Compare Kohomono phoneme inventory with related Upper Cross languages
lang_compare = {
    'Language': ['Kohomono\n(this study)','Lokaa\n(Iwara 1989)','Leggbo\n(Udoh 2014)',
                 'Mbembe\n(Akpan 2007)','Efik\n(review)','Ibibio\n(Essien)','Anaang\n(Udoh 2012)'],
    'Consonants': [24, 21, 23, 26, 20, 18, 20],
    'Vowels':     [10, 8,  7,  7,  6,  7,  7],
    'Tone_levels':[4,  2,  3,  3,  5,  3,  3],
    'Nasal_cons': [4,  3,  5,  3,  4,  4,  4],
    'Family':     ['Upper Cross','Upper Cross','Upper Cross','Upper Cross','Lower Cross','Lower Cross','Lower Cross'],
}
df_lang = pd.DataFrame(lang_compare)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
fig.patch.set_facecolor('white')

family_colors = {'Upper Cross':'#1D3557','Lower Cross':'#E63946'}
bar_fam_colors = [family_colors[f] for f in df_lang['Family']]

for ax, metric, title in zip(axes, 
    ['Consonants','Vowels','Tone_levels','Nasal_cons'],
    ['A. Consonant Phoneme Inventory Size','B. Vowel Phoneme Inventory Size',
     'C. Number of Functional Tone Levels','D. Nasal Consonant Count']):
    bars = ax.bar(range(len(df_lang)), df_lang[metric], color=bar_fam_colors,
                   edgecolor='white', linewidth=1.2, width=0.65)
    ax.set_xticks(range(len(df_lang)))
    ax.set_xticklabels(df_lang['Language'], fontsize=9)
    for bar, val in zip(bars, df_lang[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, str(val),
               ha='center', fontsize=10, fontweight='bold')
    # Highlight Kohomono
    bars[0].set_edgecolor('#F4A261')
    bars[0].set_linewidth(3)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel('Count', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('#F8F9FA')
    legend_els = [mpatches.Patch(color=v, label=k) for k,v in family_colors.items()]
    ax.legend(handles=legend_els, fontsize=9)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig8_typological_comparison.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 8")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 9: Distinctive Feature Correlation and Confusion Matrix
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

# Panel A: Feature intercorrelation
feat_df_plot = df_cons[['voiced','labial','coronal','dorsal','sonorant','continuant','nasal_feat','word_initial','word_final']]
feat_df_plot.columns = ['Voiced','Labial','Coronal','Dorsal','Sonorant','Continuant','Nasal','Initial','Final']
corr = feat_df_plot.corr()
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
#sns.heatmap(corr, ax=axes[0], annot=True, fmt='.2f', cmap='coolwarm', center=0,
#            mask=mask, linewidths=0.5, linecolor='white',
#            cbar_kws={'label':'Pearson r', 'shrink':0.8}, annot_kws={'size':9})
sns.heatmap(corr, ax=axes[0], annot=False, cmap='coolwarm', center=0,
            mask=mask, linewidths=0.8, linecolor='white',
            cbar_kws={'label':'Pearson Correlation (r)', 'shrink':0.8})

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        if not mask[i, j]:
            axes[0].text(j + 0.5, i + 0.5, f'{corr.iloc[i, j]:.2f}',
                         ha='center', va='center',
                         fontsize=10, fontweight='bold', color='black')

axes[0].set_title('A. Distinctive Feature Intercorrelations\n(Pearson r, lower triangle)', fontsize=11, fontweight='bold', pad=8)

# Panel B: Confusion matrix of RF classifier (using augmented data)
from sklearn.model_selection import cross_val_predict
np.random.seed(42)
X_aug2 = np.vstack([X + np.random.normal(0, 0.1, X.shape) for _ in range(15)])
y_aug2 = np.tile(y_enc, 15)
rf_pred = cross_val_predict(RandomForestClassifier(n_estimators=100, random_state=42),
                             X_aug2, y_aug2, cv=4)
cm = confusion_matrix(y_aug2, rf_pred)
class_names = le.classes_
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
#sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
#            xticklabels=[c.capitalize() for c in class_names],
#            yticklabels=[c.capitalize() for c in class_names],
#            linewidths=0.5, linecolor='white', cbar_kws={'label':'Proportion'})
sns.heatmap(cm_norm, annot=False, cmap='Blues', ax=axes[1],
            xticklabels=class_labels,
            yticklabels=class_labels,
            linewidths=0.8, linecolor='white',
            cbar_kws={'label':'Classification Probability'})

for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        axes[1].text(j + 0.5, i + 0.5, f'{cm_norm[i, j]:.2f}',
                     ha='center', va='center',
                     fontsize=12, fontweight='bold', color='black')

axes[1].set_title('B. Random Forest Confusion Matrix\nConsonant Manner Classification (Normalized)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xlabel('Predicted Manner', fontsize=11)
axes[1].set_ylabel('True Manner', fontsize=11)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig9_feature_corr_confusion.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 9")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 10: ATR Vowel Harmony Computational Model
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel A: ATR groupings
atr_plus = df_vowels[df_vowels['ATR']==1]
atr_minus = df_vowels[df_vowels['ATR']==0]
vowel_positions = {
    'i':(1,5),'e':(1.5,3.5),'u':(5,5),'o':(4.5,3.5),'o:':(4.7,3.7),
    'a':(3,1),'a:':(3.2,1.2),'ɛ':(1.8,2),'ɔ':(4.2,2),'ɔ:':(4.4,2.2)
}
# ATR+ zone
circle1 = plt.Circle((2.8,3.5), 2.5, color='#1D3557', fill=True, alpha=0.12, linewidth=2)
circle2 = plt.Circle((3.2,3.0), 2.8, color='#E63946', fill=True, alpha=0.10, linewidth=2, linestyle='--')
axes[0].add_patch(circle1); axes[0].add_patch(circle2)
for v, (x, y) in vowel_positions.items():
    row = df_vowels[df_vowels['phoneme']==v]
    if len(row) > 0:
        is_atr = row['ATR'].values[0]
        col = '#1D3557' if is_atr else '#E63946'
        ls = '-' if ':' not in v else '--'
        axes[0].scatter(x, y, s=350, color=col, zorder=5, edgecolors='white', linewidth=2)
        axes[0].text(x, y+0.35, f'/{v}/', ha='center', fontsize=12, fontweight='bold', color=col)
axes[0].set_xlim(0,6); axes[0].set_ylim(0,6.5)
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_title('A. ATR Vowel Harmony Sets in Kohomono\n([+ATR] blue; [-ATR] red)', fontsize=11, fontweight='bold', pad=8)
legend_els = [mpatches.Patch(color='#1D3557', label='+ATR set: /i, e, u, o, o:/'),
              mpatches.Patch(color='#E63946', label='-ATR set: /ɛ, a, a:, ɔ, ɔ:/')]
axes[0].legend(handles=legend_els, fontsize=10, loc='lower center')
axes[0].set_facecolor('#F8F9FA')

# Panel B: Vowel feature profile heatmap
vowel_feat_cols = ['ATR','high_feat','low_feat','back_feat','long_feat']
vowel_feat_labels = ['ATR','+High','+Low','+Back','+Long']
vowel_matrix = df_vowels[vowel_feat_cols].values.T
#sns.heatmap(vowel_matrix, ax=axes[1], cmap='RdYlBu_r',
#            xticklabels=df_vowels['phoneme'], yticklabels=vowel_feat_labels,
#            linewidths=0.8, linecolor='white', cbar_kws={'label':'Feature Value'},
#            annot=True, fmt='d', annot_kws={'size':11})
sns.heatmap(vowel_matrix, ax=axes[1], cmap='RdYlBu_r',
            annot=False,
            xticklabels=['/i/','/e/','/ɛ/','/a/','/a:/','/u/','/o/','/o:/','/ɔ/','/ɔ:/'],
            yticklabels=['+ATR','+High','+Low','+Back','+Long'],
            linewidths=0.8, linecolor='white',
            cbar_kws={'label':'Feature Presence (1) or Absence (0)'})

for i in range(vowel_matrix.shape[0]):
    for j in range(vowel_matrix.shape[1]):
        axes[1].text(j + 0.5, i + 0.5, str(vowel_matrix[i, j]),
                     ha='center', va='center',
                     fontsize=13, fontweight='bold', color='black')

axes[1].set_title('B. Vowel Distinctive Feature Matrix\n(10 phonemic vowels × 5 features)', fontsize=11, fontweight='bold', pad=8)
axes[1].set_xticklabels(df_vowels['phoneme'], fontsize=12)
axes[1].set_yticklabels(vowel_feat_labels, fontsize=10, rotation=0)


plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig10_vowel_harmony.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 10")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 11: Comprehensive Metrics Summary Heatmap
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('white')
summary_data = {
    'Model': ['Random Forest','Logistic Reg.','SVM (RBF)','k-NN','Grad. Boost'],
    'CV Mean': [cv_results[n].mean() for n in models],
    'CV Std':  [cv_results[n].std()  for n in models],
    'Min Fold':[cv_results[n].min()  for n in models],
    'Max Fold':[cv_results[n].max()  for n in models],
}
summary_df = pd.DataFrame(summary_data).set_index('Model')
summary_num = summary_df.copy()
#sns.heatmap(summary_num, annot=summary_df.round(3).values, fmt='', cmap='YlOrRd',
#            ax=ax, linewidths=0.5, linecolor='white',
#            cbar_kws={'label':'Accuracy Score'}, annot_kws={'size':12, 'weight':'bold'})
sns.heatmap(summary_num, annot=False, fmt='', cmap='YlOrRd',
            ax=ax, linewidths=0.8, linecolor='white',
            cbar_kws={'label':'Accuracy / Performance Metric'})

for i in range(summary_num.shape[0]):
    for j in range(summary_num.shape[1]):
        value = summary_num.iloc[i, j]
        ax.text(j + 0.5, i + 0.5, f'{value:.3f}',
                ha='center', va='center',
                fontsize=14, fontweight='bold', color='black')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, rotation=0)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig11_metrics_summary.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 11")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 12: Language Endangerment & Documentation Impact
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Panel A: Speaker population decline simulation
years = np.array([1963, 1973, 1991, 2006, 2019, 2024, 2030, 2040])
speakers = np.array([55000, 48000, 38000, 35000, 33000, 31000, 27000, 22000])
is_projected = np.array([0,0,0,0,0,0,1,1], dtype=bool)
axes[0].plot(years[~is_projected], speakers[~is_projected], 'o-', color='#1D3557', lw=2.5, 
             markersize=8, label='Documented (census/Ethnologue)')
axes[0].plot(years[is_projected], speakers[is_projected], 'o--', color='#E63946', lw=2.5,
             markersize=8, label='Projected (without intervention)')
axes[0].axvline(2020, color='#F4A261', linestyle=':', lw=2, label='This study (Etu 2020)')
axes[0].fill_between(years, speakers*0.85, speakers*1.15, alpha=0.1, color='#1D3557')
axes[0].set_title('A. Kohomono Speaker Population Trend\n(Documented decline; ~33,000 speakers in 2019)', fontsize=11, fontweight='bold', pad=8)
axes[0].set_xlabel('Year', fontsize=11); axes[0].set_ylabel('Estimated Speakers', fontsize=11)
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_facecolor('#F8F9FA'); axes[0].spines[['top','right']].set_visible(False)

# Panel B: Documentation coverage of sound system
categories = ['Phonemic\nConsonants','Vowel\nPhonemes','Tone\nTypes','Phono.\nProcesses','Syllable\nStructures','Allophones\nDocumented']
this_study = [24, 10, 4, 13, 8, 18]
cook_1969  = [23, 5, 0, 0, 0, 5]
njoku_2016 = [31, 11, 2, 3, 2, 8]
x = np.arange(len(categories))
width = 0.28
axes[1].bar(x - width, cook_1969, width, label='Cook (1969)', color='#A8DADC', edgecolor='white')
axes[1].bar(x, njoku_2016, width, label='Njoku (2016)', color='#457B9D', edgecolor='white')
axes[1].bar(x + width, this_study, width, label='Etu (2020) — This Study', color='#1D3557', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(categories, fontsize=9)
axes[1].set_title('B. Cumulative Documentation of Kohomono\nPhonological System Across Studies', fontsize=11, fontweight='bold', pad=8)
axes[1].set_ylabel('Units Documented', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].spines[['top','right']].set_visible(False)
axes[1].set_facecolor('#F8F9FA')

plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig12_endangerment_documentation.png', dpi=DPI, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved Fig 12")

print(f"\n✅ All 12 figures saved to {FIGDIR}/")
print("\n=== ML RESULTS SUMMARY ===")
for name, short in zip(models.keys(), model_names_short):
    scores = cv_results[name]
    print(f"{name:20s} | CV Mean={scores.mean():.3f} | SD={scores.std():.3f} | Range=[{scores.min():.3f}-{scores.max():.3f}]")


Data loaded. Consonants: 24 | Vowels: 10
Processes: 13 | Tone pairs: 15
Saved Fig 1
Saved Fig 2
Saved Fig 3
Saved Fig 4
Saved Fig 5
Saved Fig 6
Saved Fig 7
Saved Fig 8
Saved Fig 9
Saved Fig 10
Saved Fig 11
Saved Fig 12

✅ All 12 figures saved to kohomono_figures2/

=== ML RESULTS SUMMARY ===
Random Forest        | CV Mean=0.917 | SD=0.083 | Range=[0.833-1.000]
Logistic Reg.        | CV Mean=0.833 | SD=0.118 | Range=[0.667-1.000]
SVM (RBF)            | CV Mean=0.792 | SD=0.072 | Range=[0.667-0.833]
k-NN                 | CV Mean=0.583 | SD=0.186 | Range=[0.333-0.833]
Grad. Boost          | CV Mean=0.917 | SD=0.083 | Range=[0.833-1.000]
